# Conservare quello che e' gia' stato calcolato

*Anatomia di un LLM* · capitolo 7, Generare · script `06-02`, `scripts/06_02_kv_cache.py`

Il lavoro rifatto a ogni carattere non cambia mai, perche' la maschera causale
impedisce al passato di dipendere dal futuro. Se non cambia, si puo' tenere da
parte: e' la KV cache, e trasforma un costo quadratico in uno lineare.

Si esegue con `python3 scripts/06_02_kv_cache.py`
e dura circa 10 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

> **Questo notebook usa il modello addestrato.** Se `checkpoints/agogpt.pt` non c'è, la seconda cella ne scarica i pesi; per addestrarlo da te c'è `05_02_pretrain_agogpt`.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/06_02_kv_cache.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [ ]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [ ]:
# Questo notebook usa il modello addestrato. Se non c'e', si scaricano i
# suoi pesi (7 MB) invece di addestrarlo: per addestrarlo davvero c'e' il
# notebook 05_02_pretrain_agogpt, che dura alcuni minuti.
import urllib.request

modello = Path("checkpoints") / "agogpt.pt"
if not modello.exists():
    modello.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://github.com/fcollova/llm-anatomy/releases/download/modello-v0.1/agogpt-pesi.pt", modello)

In [ ]:
import time

import torch

from llm_anatomy.config import AgoGPTConfig
from llm_anatomy.data import tokenizer_corpus
from llm_anatomy.model import AgoGPT
from llm_anatomy.sample import genera
from llm_anatomy.train import carica_checkpoint
from llm_anatomy.utils import (byte_leggibili, dec, percorso, prepara,
                               tabella, titolo, verifica)

prepara()
torch.set_num_threads(1)
print("[misure] torch.set_num_threads(1) per rendere i tempi confrontabili")

file_ckpt = percorso("checkpoints", "agogpt.pt")
if not file_ckpt.exists():
    raise SystemExit("checkpoint assente: eseguire prima 05-02")
stato = carica_checkpoint(file_ckpt)
cfg = AgoGPTConfig(**stato["cfg"])
modello = AgoGPT(cfg)
modello.load_state_dict(stato["modello"])
modello.eval()
tok = tokenizer_corpus()

avvio = torch.tensor([tok.encode("Risotto coi funghi\n\n")])

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337
[misure] torch.set_num_threads(1) per rendere i tempi confrontabili


In [ ]:
titolo("Cio' che non cambia si conserva")
print("  chiavi e valori di un token dipendono solo da quel token e dalla sua")
print("  posizione: una volta calcolati non cambiano piu'. La cache li tiene,")
print("  e a ogni passo il modello elabora un solo token nuovo invece di tutto")
print("  il prefisso")


Cio' che non cambia si conserva
-------------------------------
  chiavi e valori di un token dipendono solo da quel token e dalla sua
  posizione: una volta calcolati non cambiano piu'. La cache li tiene,
  e a ogni passo il modello elabora un solo token nuovo invece di tutto
  il prefisso


In [ ]:
titolo("Stesso testo, cifra per cifra")
for temperatura, etichetta in ((0.0, "greedy"), (0.8, "con campionamento")):
    g1 = torch.Generator().manual_seed(1234)
    g2 = torch.Generator().manual_seed(1234)
    con = genera(modello, avvio.clone(), 60, temp=temperatura, usa_cache=True,
                 generatore=g1)
    senza = genera(modello, avvio.clone(), 60, temp=temperatura, usa_cache=False,
                   generatore=g2)
    uguali = torch.equal(con, senza)
    print(f"  {etichetta:<20} identici: {uguali}")
    verifica(uguali, f"la cache non cambia il risultato ({etichetta})")
print("  con il campionamento la verifica e' piu' forte: gli stessi numeri")
print("  casuali devono cadere sugli stessi logit, altrimenti le due sequenze")
print("  divergerebbero al primo scarto numerico")


Stesso testo, cifra per cifra
-----------------------------


  greedy               identici: True
  [verifica superata] la cache non cambia il risultato (greedy)


  con campionamento    identici: True
  [verifica superata] la cache non cambia il risultato (con campionamento)
  con il campionamento la verifica e' piu' forte: gli stessi numeri
  casuali devono cadere sugli stessi logit, altrimenti le due sequenze
  divergerebbero al primo scarto numerico


In [ ]:
titolo("Quanto si guadagna")
def cronometra(n_token: int, cache: bool, ripetizioni: int = 3) -> float:
    tempi = []
    for _ in range(ripetizioni):
        t0 = time.perf_counter()
        genera(modello, avvio.clone(), n_token, temp=0.0, usa_cache=cache)
        tempi.append(time.perf_counter() - t0)
    tempi.sort()
    return tempi[len(tempi) // 2]


righe = []
for n in (16, 32, 64, 100):
    t_con = cronometra(n, True)
    t_senza = cronometra(n, False)
    righe.append([n, dec(t_senza, 3), dec(t_con, 3), dec(t_senza / t_con, 2) + "x",
                  dec(1000 * t_con / n, 2)])
tabella(["token generati", "senza cache (s)", "con cache (s)", "speedup",
         "ms per token con cache"], righe)
guadagni = [float(r[3][:-1].replace(",", ".")) for r in righe]
verifica(guadagni[-1] > guadagni[0],
         "il vantaggio della cache cresce con la lunghezza della sequenza")
print("  lo speedup non e' una costante: cresce, perche' e' il rapporto fra un")
print("  costo quadratico e uno lineare")


Quanto si guadagna
------------------


token generati  senza cache (s)  con cache (s)  speedup  ms per token con cache
--------------  ---------------  -------------  -------  ----------------------
16                        0,043          0,017    2,50x                    1,08
32                        0,083          0,033    2,54x                    1,02
64                        0,214          0,063    3,42x                    0,98
100                       0,407          0,105    3,88x                    1,05
  [verifica superata] il vantaggio della cache cresce con la lunghezza della sequenza
  lo speedup non e' una costante: cresce, perche' e' il rapporto fra un
  costo quadratico e uno lineare


In [ ]:
titolo("Il prezzo si paga in memoria")
caches = modello.nuove_cache()
idx = avvio.clone()
righe = []
with torch.no_grad():
    offset = 0
    for passo in range(cfg.block_size - avvio.shape[1]):
        ingresso = idx if offset == 0 else idx[:, -1:]
        logits, _ = modello(ingresso, caches=caches, pos_offset=offset)
        offset += ingresso.shape[1]
        idx = torch.cat([idx, logits[:, -1:].argmax(dim=-1)], dim=1)
        if offset in (18, 32, 64, 96, 128):
            byte = sum(c.byte() for c in caches)
            righe.append([offset, byte_leggibili(byte),
                          byte_leggibili(byte / offset)])
tabella(["token in cache", "memoria totale", "per token"], righe)

per_token = sum(c.byte() for c in caches) / offset
print(f"  la crescita e' lineare: {byte_leggibili(per_token)} per token, cioe'")
print(f"  2 tensori x {cfg.n_layer} blocchi x {cfg.n_kv_head} teste x "
      f"{cfg.d_head} dimensioni x 4 byte")


Il prezzo si paga in memoria
----------------------------
token in cache  memoria totale  per token
--------------  --------------  ---------
32                   192.0 KiB    6.0 KiB
64                   384.0 KiB    6.0 KiB
96                   576.0 KiB    6.0 KiB
  la crescita e' lineare: 6.0 KiB per token, cioe'
  2 tensori x 4 blocchi x 6 teste x 32 dimensioni x 4 byte


In [ ]:
titolo("Che cosa significa su un modello vero")
righe = []
for nome, n_layer, kv, d_head, contesto in (
    ("mini-modello, ctx 128", cfg.n_layer, cfg.n_kv_head, cfg.d_head, 128),
    ("7B GQA a 8, ctx 8k", 32, 8, 128, 8192),
    ("7B GQA a 8, ctx 128k", 32, 8, 128, 131072),
):
    byte_token = 2 * n_layer * kv * d_head * (4 if "mini" in nome else 2)
    righe.append([nome, byte_leggibili(byte_token),
                  byte_leggibili(byte_token * contesto)])
tabella(["configurazione", "per token", "a contesto pieno"], righe)
print("  a contesto lungo la cache diventa piu' grande dei pesi stessi: e' il")
print("  motivo per cui esistono GQA, la quantizzazione della cache e le")
print("  finestre scorrevoli. Il problema non e' piu' generare, e' ricordare")


Che cosa significa su un modello vero
-------------------------------------
configurazione         per token  a contesto pieno
---------------------  ---------  ----------------
mini-modello, ctx 128    6.0 KiB         768.0 KiB
7B GQA a 8, ctx 8k     128.0 KiB           1.0 GiB
7B GQA a 8, ctx 128k   128.0 KiB          16.0 GiB
  a contesto lungo la cache diventa piu' grande dei pesi stessi: e' il
  motivo per cui esistono GQA, la quantizzazione della cache e le
  finestre scorrevoli. Il problema non e' piu' generare, e' ricordare


## Da provare

misura la memoria della cache a contesto 512.